# Phase 4: Model Training and Evaluation

This notebook compares exactly three traditional algorithms under fair experimental conditions, tunes hyperparameters, and evaluates the final selected model on a held-out test set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Add src to path so we can import our modules
sys.path.append(os.path.abspath('..'))
from src.preprocessing import prepare_features
from src.modeling import compare_models, tune_hyperparameters, evaluate_model

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

## 1. Load Processed Data

We load the cleaned data and the preprocessor pipeline. 

In [ ]:
# Load the cleaned dataset
df_clean = pd.read_csv('../data/processed/cleaned_data.csv')

# Load the preprocessor
preprocessor = joblib.load('../models/preprocessor.pkl')

# Extract X and y (y is log1p of Price)
X_raw, y = prepare_features(df_clean)

## 2. Train-Test Split & Preprocessing

We split the data 80% for training and 20% for final testing.
**Crucially, we only fit the preprocessor on the training data** to prevent data leakage from the test set. The test set is only transformed.

In [ ]:
# Split data (80/20)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# Fit preprocessor on training data and transform both
X_train = preprocessor.fit_transform(X_train_raw, y_train)
X_test = preprocessor.transform(X_test_raw)

print(f"Training set: {X_train.shape[0]} records")
print(f"Testing set: {X_test.shape[0]} records")

## 3. Three-Algorithm Baseline Comparison

In accordance with the PDF requirements, we compare three distinct algorithms:
1. **Linear Regression** (Parametric baseline)
2. **Decision Tree Regressor** (Non-linear, single tree)
3. **Random Forest Regressor** (Ensemble bagging)

We use 5-Fold Cross Validation.

In [ ]:
# Initialize baseline models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1)
}

# Run cross-validation comparison (Metrics on Log scale)
cv_results = compare_models(models, X_train, y_train, cv=5)
print("Cross-Validation Results (Log Scale):")
cv_results

**Finding 3.1:** The Random Forest Regressor typically outperforms the others by having the lowest RMSE and highest R² during cross-validation. Decision Trees tend to overfit, and Linear Regression may struggle with non-linear interactions (e.g., location value vs floor area).

## 4. Hyperparameter Tuning

We'll tune the Random Forest since it's the strongest candidate. 
*(Note: If Decision Tree or Linear Regression were surprisingly better, we would tune them instead. We tune the most promising model based on CV).*

In [ ]:
print("Tuning Random Forest...")
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

best_rf_model = tune_hyperparameters(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    X_train,
    y_train,
    cv=5
)

## 5. Final Test Set Evaluation

We evaluate the tuned model on the 20% untouched test set.
The metrics will be converted back to the original PHP scale so they are interpretable.

In [ ]:
# Evaluate the best model
test_metrics = evaluate_model(best_rf_model, X_test, y_test, is_log_target=True)

print("\nFinal Test Set Performance (Original PHP Scale):")
for metric, value in test_metrics.items():
    if metric == 'R2':
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: PHP {value:,.2f}")

## 6. Error Analysis & Residuals

Let's visualize the model's predictions vs actual prices on the test set.

In [ ]:
y_test_orig = np.expm1(y_test)
y_pred_log = best_rf_model.predict(X_test)
y_pred_orig = np.expm1(y_pred_log)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
ax1.scatter(y_test_orig, y_pred_orig, alpha=0.5, color='blue')
ax1.plot([y_test_orig.min(), y_test_orig.max()], [y_test_orig.min(), y_test_orig.max()], 'r--', lw=2)
ax1.set_xlabel('Actual Price (PHP)')
ax1.set_ylabel('Predicted Price (PHP)')
ax1.set_title('Actual vs Predicted Prices')

# Residuals Distribution
residuals = y_test_orig - y_pred_orig
sns.histplot(residuals, kde=True, ax=ax2, color='orange')
ax2.set_xlabel('Residual Error (PHP)')
ax2.set_title('Distribution of Residuals')

plt.tight_layout()
plt.savefig('../documentation/screenshots/residuals.png')

**Finding 6.1:** 
The Actual vs Predicted plot shows that the model tracks prices reasonably well across the main market segment. The residuals are centered around zero, though the variance in error increases for higher-priced luxury properties, which is typical for real estate valuation.

## 7. Save Final Model

We save the fully tuned and fitted Random Forest model. The Streamlit app will load both this model and the `preprocessor.pkl`.

In [ ]:
# Save the best model
model_path = '../models/best_model.pkl'
joblib.dump(best_rf_model, model_path)
print(f"Saved final selected model to {model_path}")

# Note: We must also save the preprocessor that was fit ONLY on the training data.
# This prevents data leakage in the deployment pipeline.
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("Saved training-fit preprocessor to ../models/preprocessor.pkl")